# StereoQueerEval 2027 — Task B: Role-Aware Conditional Context Retrieval (RACC)

Notebook này cung cấp quy trình nghiên cứu, thực nghiệm và đánh giá toàn diện cho kiến trúc **RACC (Role-Aware Conditional Context Retrieval)** dành cho **Task B (Hate Speech Classification: 3 classes `no`, `yes_implicit`, `yes_explicit`)**.

```
                                 [INPUT SEQUENCE]
               Comment: <C> ... </C> | Title: <T> ... </T> | Description: <D> ... </D>
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ mmBERT BACKBONE & MULTI-SCALE PROGRESSIVE EXTRACTION                                    │
│   Input IDs + Attention Mask ──► mmBERT-base (22 Layers, ModernBERT RoPE)               │
│                                ├──► H20 (idx -3): Early Contextual Representations      │
│                                ├──► H21 (idx -2): Mid-Level Contextual Representations  │
│                                └──► H22 (idx -1): Top Global Representations            │
│   Role Embeddings E_role (padding_idx=0, std=0.02) ──► LayerNorm(H_k + E_role)          │
└───────────────────────────────────────┬─────────────────────────────────────────────────┘
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 1: COMMENT GROUNDING (Hop 1 on H20)                                               │
│   q_0 ∈ [3, 768] queries Comment Tokens under M_comment                                 │
│   Q_C = q_0 + Dropout(CrossAttn(LN(q_0), H20_role, mask=M_comment))                     │
└───────────────────────────────────────┬─────────────────────────────────────────────────┘
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 2: CONDITIONAL CONTEXT RETRIEVAL & RELATIONAL GATED FUSION (Hop 2 on H21)         │
│   r_T = CrossAttn(LN_T(Q_C), H21_role, mask=M_title)                                    │
│   r_D = CrossAttn(LN_D(Q_C), H21_role, mask=M_desc)                                     │
│   Adaptive Gating: [alpha_T, alpha_D] = Softmax(f([LN_gate(Q_C); r_T]), f([LN_gate; r_D]))│
│   r_ctx = alpha_T * r_T + alpha_D * r_D                                                 │
│   Relational Matching: u = [Q_C; r_ctx; Q_C * r_ctx; |Q_C - r_ctx|]                     │
│   g = Sigmoid(MLP(u)),  Q_ctx = LayerNorm(Q_C + g * W_r(r_ctx))                         │
└───────────────────────────────────────┬─────────────────────────────────────────────────┘
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 3: GLOBAL VERIFICATION (Hop 3 on H22)                                             │
│   Q_G = Q_ctx + Dropout(CrossAttn(LN(Q_ctx), H22_role, mask=M_all))                      │
└───────────────────────────────────────┬─────────────────────────────────────────────────┘
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 4: QUERY INTERACTION (MHSA) & TASK-SPECIFIC FFN REFINEMENT                        │
│   Z = LayerNorm(Q_G + MHSA(Q_G, Q_G, Q_G)) ──► FFN Refinement ──► Z ∈ [B, 3, 768]     │
└───────────────────────────────────────┬─────────────────────────────────────────────────┘
                                        │
                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 5: CLASSIFICATION HEAD & TASK C REPRESENTATION BRIDGE                             │
│   Logits s = Linear([z_no; z_implicit; z_explicit]) ∈ [B, 3]                            │
│   Task C Bridge: h_B = ∑_c p_c * z_c ∈ [B, 768]                                         │
└─────────────────────────────────────────────────────────────────────────────────────────┘
```

---
### Bảng Giả Thuyết Khoa Học (Hypothesis Testing Table):
| Mã | Giả thuyết | Baseline đối chứng | Mục tiêu chứng minh |
|:---|:---|:---|:---|
| **H1** | Role-restricted Comment Grounding | B2 (`prog_h20_full`) | Giới hạn tầm nhìn ở Hop 1 vào Comment ngăn ngừa nhiễu ngữ cảnh sớm |
| **H2** | Tách độc lập Title & Description + Adaptive Gating | B3 (`fixed_h22_role`) | Ngăn Title lấn át Description khi Description chứa thông tin ngầm định |
| **H3** | Khai thác Multi-scale H20 $\to$ H21 $\to$ H22 | B4 (`fixed_h22_cond`) | Tận dụng biểu diễn phân cấp cục bộ $\to$ toàn cục của mmBERT |
| **H4** | ESIM/BiDAF Relational Gated Fusion | Additive Fusion | Đánh giá tương tác quan hệ $|Q_C - r_{\text{ctx}}|$ và $Q_C \odot r_{\text{ctx}}$ |
| **H5** | Query Interaction MHSA | `use_query_interaction=False` | Tương quan phân loại đồng thời giữa Implicit và Explicit |

## 1. Clone hoặc Cập nhật Mã Nguồn từ GitHub

Thay thế URL repository của bạn ở bên dưới nếu đang chạy trên Google Colab / Kaggle / Cloud GPU server.

In [ ]:
# Khai báo thông tin repository (thay bằng URL GitHub của bạn)
import os
import sys

REPO_URL = "https://github.com/your-username/stereoqueer-pipeline.git"
REPO_DIR = "stereoqueer-pipeline"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    print(f"Repository directory '{REPO_DIR}' already exists.")
    if os.path.isdir(REPO_DIR):
        %cd {REPO_DIR}
        !git pull

# Thêm root vào sys.path để import pipeline package
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current Working Directory:", os.getcwd())
!ls -la

## 2. Thiết lập Môi trường & Cài đặt Thư viện (Hardware & Dependencies)

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("=== HARDWARE & ENVIRONMENT INSPECTION ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Current Allocated VRAM: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    # Check Fast-path SDPA & FlashAttention an toàn
    has_sdpa = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
    print(f"PyTorch SDPA Available: {has_sdpa}")
    if hasattr(torch.backends.cuda, 'flash_sdp_enabled'):
        print(f"FlashAttention SDP Enabled: {torch.backends.cuda.flash_sdp_enabled()}")
    if hasattr(torch.backends.cuda, 'mem_efficient_sdp_enabled'):
        print(f"Mem Efficient SDP Enabled: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
else:
    print("Running on CPU / MPS fallback.")

# Set seeds for reproducible research
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Kiểm tra Đơn vị (Unit Tests & Smoke Test Verification)
Chạy smoke test để xác thực `TaskBRACCModel`, các baseline modes (`racc`, `fixed_h22_full`, `prog_h20_full`, `fixed_h22_role`, `fixed_h22_cond`), dynamic `multiscale_layer_indices`, và `need_weights=False` fast-path.

In [ ]:
!python smoke_test.py

## 4. Tự Động Tìm & Chuẩn Bị Dữ Liệu TSV (Kaggle / Local / Synthetic Fallback)
Tự động dò tìm các đường dẫn chứa tập dữ liệu TSV đa ngữ của StereoQueerEval (`EN`, `IT`, `NL`).

In [ ]:
import os
import glob

os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints_racc", exist_ok=True)

# 1. Danh sách đường dẫn tìm kiếm phổ biến (Kaggle / Colab / Local)
kaggle_paths = [
    "/kaggle/input/datasets/amnpham/lgbtqia/LGBT",
    "/kaggle/input/lgbtqia/LGBT",
    "/kaggle/input/stereoqueereval",
    "/kaggle/input",
    "data"
]

tsv_files = []
for kp in kaggle_paths:
    if os.path.exists(kp):
        found = glob.glob(f"{kp}/**/*.tsv", recursive=True) + glob.glob(f"{kp}/*.tsv")
        if found:
            tsv_files.extend(found)
            break

# 2. Nếu chưa có file nào, tự động sinh sample data để test pipeline
if not tsv_files:
    print("Chưa tìm thấy file TSV chính thức. Đang sinh dữ liệu mẫu để kiểm thử...")
    !python generate_sample_data.py
    tsv_files = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".tsv")]

tsv_files = sorted(list(set(tsv_files)))
print(f"Đã tìm thấy {len(tsv_files)} file TSV:")
for f in tsv_files:
    print(" -", f)

## 5. Load Dữ Liệu & Phân Chia Anti-Leakage (`GroupShuffleSplit` / Video Level)
Đảm bảo các comment từ cùng một video YouTube (`yt_title`) không bị rò rỉ giữa tập Train và tập Validation.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from pipeline.config import PipelineConfig, HATE_CLASSES, HATE2IDX, IDX2HATE
from pipeline.data import DataPipeline, ROLE_PAD, ROLE_TITLE, ROLE_DESC, ROLE_COMMENT, NUM_ROLES
from pipeline.utils import set_seed, get_seeded_generator, seed_worker

config = PipelineConfig(
    task="stereoqueer",
    target_task="hs", # Task B (Hate Speech)
    model_type="task_b_racc",
    mmbert_model_name="jhu-clsp/mmbert-base",
    batch_size=32,
    max_length=256,
    multiscale_layer_indices=[-3, -2, -1], # H20 (idx -3), H21 (idx -2), H22 (idx -1)
    decoder_ffn_dim=1536,
    use_query_interaction=True,
    use_hierarchical_prediction=True,
    hierarchical_threshold=0.50,
    dropout=0.15,
    weight_decay=0.01,
    learning_rate=2e-4,
    loss_type="focal",
    focal_gamma=2.0,
    label_smoothing=0.05,
    two_phase=True,
    freeze_phase_epochs=3,
    unfreeze_phase_epochs=12,
    unfreeze_layers=3,
    unfreeze_lr=1.5e-5,
    head_unfreeze_lr=5e-5,
    patience=5,
    output_dir="checkpoints_racc"
)

set_seed(config.random_seed)
data_pipeline = DataPipeline(config)
df_all = data_pipeline.load_data(tsv_files)
df_train, df_val = data_pipeline.split_data(test_size=0.15, random_state=config.random_seed)

print(f"Train samples: {len(df_train)} | Val samples: {len(df_val)}")
print("\nPhân bố nhãn Task B trên tập Train:")
print(df_train['hate_speech'].value_counts())
print("\nPhân bố nhãn Task B trên tập Val:")
print(df_val['hate_speech'].value_counts())

# Balanced Class Weights để chống sụp đổ dự đoán
train_hs_labels = df_train['hate_speech'].map(HATE2IDX).values
weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=train_hs_labels)
config.class_weights = weights.tolist()
print("\nBalanced Class Weights [no, yes_implicit, yes_explicit]:", [round(w, 3) for w in config.class_weights])

## 6. Tokenizer & Tạo Role-Aware DataLoader (`TaskBRoleDataset`)
Tokenize các đoạn văn bản độc lập và gán `role_ids` (`ROLE_PAD=0`, `ROLE_TITLE=1`, `ROLE_DESC=2`, `ROLE_COMMENT=3`).

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from pipeline.task_b_data import TaskBRoleDataset

print(f"Loading mmBERT tokenizer: {config.mmbert_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(config.mmbert_model_name)

train_ds = TaskBRoleDataset(df_train, tokenizer, max_len=config.max_length)
val_ds = TaskBRoleDataset(df_val, tokenizer, max_len=config.max_length)

train_loader = DataLoader(
    train_ds,
    batch_size=config.batch_size,
    shuffle=True,
    generator=get_seeded_generator(config.random_seed),
    worker_init_fn=seed_worker,
    pin_memory=torch.cuda.is_available(),
    num_workers=2
)
val_loader = DataLoader(
    val_ds,
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=2
)

sample_input_ids, sample_mask, sample_roles, sample_idx, sample_st, sample_hs, sample_tg = train_ds[0]
print("Sample Input IDs shape:", sample_input_ids.shape)
print("Sample Role IDs shape:", sample_roles.shape)
print("Role IDs distribution in sample:", torch.bincount(sample_roles))
print("Sample Gold HS Label:", sample_hs.item(), f"({IDX2HATE[sample_hs.item()]})")

# Kiểm tra giải mã token
tokens = tokenizer.convert_ids_to_tokens(sample_input_ids[:20])
roles = sample_roles[:20].tolist()
role_names = {ROLE_PAD: 'PAD/SEP', ROLE_TITLE: 'TITLE', ROLE_DESC: 'DESC', ROLE_COMMENT: 'COMMENT'}
print("\nFirst 20 tokens with role mapping:")
for t, r in zip(tokens, roles):
    print(f"  {t:<15} -> {role_names.get(r, 'UNK')}")

## 7. Khởi Tạo Mô Hình RACC (`TaskBRACCModel`)
Khởi tạo mô hình RACC với mmBERT backbone và các thành phần cốt lõi:
- `role_embeddings`: `padding_idx=ROLE_PAD`, `std=0.02`
- `context_gate`: `ContextSourceGate` với chuẩn hóa query đối xứng `context_gate_norm_q`
- `relational_fusion`: `RelationalGatedFusion` theo phong cách ESIM/BiDAF
- `_safe_cross_attention`: Cơ chế bảo vệ NaN cho empty masks + Fused SDPA Fast-Path (`need_weights=False`)

In [ ]:
from transformers import AutoModel
from pipeline.models.task_b_racc import TaskBRACCModel

print(f"Loading mmBERT backbone: {config.mmbert_model_name}...")
backbone = AutoModel.from_pretrained(config.mmbert_model_name)

# Khởi tạo Full RACC Model
racc_model = TaskBRACCModel(
    mmbert_backbone=backbone,
    num_classes=3,
    hidden_dim=config.mmbert_dim,
    num_heads=8,
    ffn_dim=config.decoder_ffn_dim,
    dropout=config.dropout,
    use_query_interaction=config.use_query_interaction,
    baseline_mode="racc",
    multiscale_layer_indices=tuple(config.multiscale_layer_indices)
)

total_params = sum(p.numel() for p in racc_model.parameters())
trainable_params = sum(p.numel() for p in racc_model.parameters() if p.requires_grad)
racc_head_params = sum(p.numel() for n, p in racc_model.named_parameters() if not n.startswith('mmbert.'))

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"RACC Multi-Hop Head Parameters: {racc_head_params:,}")

# Chạy kiểm tra Forward pass với Diagnostics
racc_model.eval()
with torch.no_grad():
    b_ids = sample_input_ids.unsqueeze(0)
    b_mask = sample_mask.unsqueeze(0)
    b_roles = sample_roles.unsqueeze(0)
    logits, h_b, diag = racc_model(b_ids, b_mask, b_roles, return_diagnostics=True)

print("\n--- FORWARD DIAGNOSTICS TEST ---")
print("Logits shape:", logits.shape)
print("Task C Bridge h_B shape:", h_b.shape)
print("Context Source Gate alpha shape:", diag['alpha_gate'].shape)
print("Sample Title vs Desc Gating weights:\n", diag['alpha_gate'][0].cpu().numpy().round(3))

## 8. Huấn Luyện 2-Phase (Two-Phase Fine-Tuning Protocol)
- **Phase 1 (Frozen Backbone)**: Huấn luyện RACC Multi-Hop Heads, ContextSourceGate và Relational Fusion (`lr=2e-4`).
- **Phase 2 (Discriminative Fine-Tuning)**: Mở khóa 3 tầng cuối của mmBERT ($H_{20}, H_{21}, H_{22}$) với tốc độ học nhỏ (`lr=1.5e-5`), giữ các tầng đầu đóng băng để duy trì tri thức nền tảng.

In [ ]:
from pipeline.task_b_trainer import TaskBTrainer

# Khởi tạo Trainer
trainer = TaskBTrainer(
    model=racc_model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    df_val=df_val
)

print("Starting Two-Phase Training for RACC...")
history = trainer.train()

# Vẽ biểu đồ hội tụ Loss & Macro-F1
if history:
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot([h['epoch'] for h in history], [h['train_loss'] for h in history], label='Train Loss', marker='o')
    plt.plot([h['epoch'] for h in history], [h['val_loss'] for h in history], label='Val Loss', marker='s')
    plt.title('Loss Convergence (Two-Phase)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot([h['epoch'] for h in history], [h['macro_f1'] for h in history], label='Macro-F1', color='green', marker='o')
    if 'f1_implicit' in history[0]:
        plt.plot([h['epoch'] for h in history], [h['f1_implicit'] for h in history], label='Implicit-F1', color='purple', linestyle='--', marker='^')
    plt.title('Validation F1 Evolution')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

## 9. Thực Nghiệm Kiểm Chứng Các Giả Thuyết (Systematic Ablation Study B1–B5)
Chạy và so sánh trực tiếp 5 biến thể kiến trúc để chứng minh tính đóng góp của từng module:
- **B1 (`fixed_h22_full`)**: $H_{22} \to H_{22} \to H_{22}$ với Full Visibility (Không role masking, không conditional context split)
- **B2 (`prog_h20_full`)**: $H_{20} \to H_{21} \to H_{22}$ với Full Visibility (Kiểm chứng H1: vai trò của Role Masking ở Hop 1)
- **B3 (`fixed_h22_role`)**: $H_{22} \to H_{22} \to H_{22}$ với Role Masking (Kiểm chứng H2: vai trò của Conditional Context Retrieval)
- **B4 (`fixed_h22_cond`)**: $H_{22} \to H_{22} \to H_{22}$ với Conditional Context Split (Kiểm chứng H3: vai trò của Multi-scale Extraction)
- **B5 (`racc`)**: Full Dual-Axis RACC (Đầy đủ cả 2 trục Multi-scale & Role-Aware)

In [ ]:
def evaluate_ablation_variant(variant_name, baseline_mode, use_qi=True, multiscale_indices=(-3, -2, -1)):
    print(f"\nEvaluating Variant: {variant_name} (mode={baseline_mode}, QI={use_qi}, indices={multiscale_indices})...")
    
    model_var = TaskBRACCModel(
        mmbert_backbone=backbone,
        num_classes=3,
        hidden_dim=config.mmbert_dim,
        num_heads=8,
        ffn_dim=config.decoder_ffn_dim,
        dropout=0.1,
        use_query_interaction=use_qi,
        baseline_mode=baseline_mode,
        multiscale_layer_indices=multiscale_indices
    )
    
    var_trainer = TaskBTrainer(
        model=model_var,
        config=config,
        train_loader=train_loader,
        val_loader=val_loader,
        df_val=df_val
    )
    
    # Huấn luyện nhanh 3 epochs để thu thập metrics đối chứng
    opt = var_trainer.build_optimizer(lr=2e-4)
    for ep in range(3):
        var_trainer.train_epoch(opt)
    
    val_loss, macro_f1, f1_dict, preds, _ = var_trainer.eval_epoch()
    return {
        'Variant': variant_name,
        'Baseline Mode': baseline_mode,
        'Query Interaction': use_qi,
        'Macro-F1': macro_f1,
        'F1(no)': f1_dict.get('no', 0.0),
        'F1(implicit)': f1_dict.get('yes_implicit', 0.0),
        'F1(explicit)': f1_dict.get('yes_explicit', 0.0)
    }

ablation_results = []
ablation_configs = [
    ("B1: Fixed H22 Full Seq", "fixed_h22_full", True, (-1, -1, -1)),
    ("B2: Prog H20-22 Full Seq", "prog_h20_full", True, (-3, -2, -1)),
    ("B3: Fixed H22 Role Masked", "fixed_h22_role", True, (-1, -1, -1)),
    ("B4: Fixed H22 Cond Context", "fixed_h22_cond", True, (-1, -1, -1)),
    ("B5: Full RACC (No QI)", "racc", False, (-3, -2, -1)),
    ("B5: Full RACC (Proposed)", "racc", True, (-3, -2, -1)),
]

for name, mode, qi, indices in ablation_configs:
    res = evaluate_ablation_variant(name, mode, qi, indices)
    ablation_results.append(res)

df_ablation = pd.DataFrame(ablation_results)
print("\n=== ABLATION STUDY COMPARATIVE RESULTS ===")
print(df_ablation.to_string(index=False))

# Vẽ biểu đồ so sánh F1
plt.figure(figsize=(10, 5))
x = np.arange(len(df_ablation))
width = 0.35
plt.bar(x - width/2, df_ablation['Macro-F1'], width, label='Macro-F1', color='royalblue')
plt.bar(x + width/2, df_ablation['F1(implicit)'], width, label='F1(Implicit)', color='coral')
plt.xticks(x, df_ablation['Variant'], rotation=25, ha='right')
plt.ylabel('F1 Score')
plt.title('Ablation Comparison Across RACC Structural Hypotheses')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 10. Phân Tích Lỗi Chuyên Sâu (Directional Error Taxonomy: Groups A–F)
Phân chia toàn bộ lỗi dự đoán thành 6 nhóm chuyển dịch để làm rõ decision boundary:
- **Group A (`no` $\to$ `implicit`)**: False Positive do counter-speech hoặc trích dẫn từ ngữ kích động.
- **Group B (`no` $\to$ `explicit`)**: False Positive do từ ngữ xúc phạm không hướng tới nhóm mục tiêu.
- **Group C (`implicit` $\to$ `no`)**: False Negative do dog whistle, mỉa mai, tiền giả định thực dụng.
- **Group D (`implicit` $\to$ `explicit`)**: Phán đoán sai mức độ nghiêm trọng do ẩn dụ mạnh.
- **Group E (`explicit` $\to$ `no`)**: False Negative do biến thể ngôn ngữ / tiếng lóng địa phương.
- **Group F (`explicit` $\to$ `implicit`)**: Giảm cấp độ nghiêm trọng của thù ghét trực tiếp.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Đánh giá mô hình RACC tốt nhất
val_loss, macro_f1, f1_dict, y_pred, y_prob = trainer.eval_epoch()
y_true = df_val['hate_speech'].tolist()
labels = ['no', 'yes_implicit', 'yes_explicit']

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'RACC Task B Confusion Matrix (Macro-F1: {macro_f1:.4f})')
plt.xlabel('Predicted')
plt.ylabel('True Gold Label')
plt.tight_layout()
plt.show()

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, target_names=labels, digits=4))

# 2. Phân vùng lỗi chuyển dịch Directional Error
df_analysis = df_val.copy()
df_analysis['predicted_label'] = y_pred
df_analysis['prob_no'] = y_prob[:, 0]
df_analysis['prob_implicit'] = y_prob[:, 1]
df_analysis['prob_explicit'] = y_prob[:, 2]

error_buckets = {
    'A: no -> implicit (FP Implicit)': df_analysis[(df_analysis['hate_speech'] == 'no') & (df_analysis['predicted_label'] == 'yes_implicit')],
    'B: no -> explicit (FP Explicit)': df_analysis[(df_analysis['hate_speech'] == 'no') & (df_analysis['predicted_label'] == 'yes_explicit')],
    'C: implicit -> no (FN Implicit)': df_analysis[(df_analysis['hate_speech'] == 'yes_implicit') & (df_analysis['predicted_label'] == 'no')],
    'D: implicit -> explicit (Misjudged)': df_analysis[(df_analysis['hate_speech'] == 'yes_implicit') & (df_analysis['predicted_label'] == 'yes_explicit')],
    'E: explicit -> no (FN Explicit)': df_analysis[(df_analysis['hate_speech'] == 'yes_explicit') & (df_analysis['predicted_label'] == 'no')],
    'F: explicit -> implicit (Downgraded)': df_analysis[(df_analysis['hate_speech'] == 'yes_explicit') & (df_analysis['predicted_label'] == 'yes_implicit')],
}

print("\n=== DIRECTIONAL ERROR DISTRIBUTION ===")
for b_name, b_df in error_buckets.items():
    print(f"{b_name}: {len(b_df)} cases ({len(b_df)/len(df_analysis)*100:.1f}%)")

# In các mẫu lỗi trong Group C quan trọng
group_c = error_buckets['C: implicit -> no (FN Implicit)']
if len(group_c) > 0:
    print("\n--- TOP SAMPLES IN GROUP C (Implicit Hate Misclassified as Non-Hate) ---")
    cols = ['yt_comment', 'yt_title', 'prob_no', 'prob_implicit'] if 'yt_title' in group_c.columns else ['yt_comment', 'prob_no', 'prob_implicit']
    print(group_c[cols].head(3).to_string())

## 11. Trực Quan Hóa Khả Năng Giải Thích (Interpretability & Gating Weights Analysis)
Phân tích cơ chế thích ứng của `ContextSourceGate`:
1. Phân bố trọng số $\alpha_{\text{title}}$ so với $\alpha_{\text{desc}}$ theo từng loại nhãn thù ghét.
2. Trực quan hóa bản đồ chú ý (Multi-Hop Attention Map) qua 3 chặng truy vấn.

In [ ]:
racc_model.eval()
all_alphas = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        ids, mask, roles, _, _, hs_labels, _ = batch
        ids = ids.to(trainer.device)
        mask = mask.to(trainer.device)
        roles = roles.to(trainer.device)
        _, _, diag = racc_model(ids, mask, roles, return_diagnostics=True)
        if diag and 'alpha_gate' in diag:
            alpha_implicit = diag['alpha_gate'][:, 1, :].cpu().numpy() # [B, 2] (Title, Desc)
            all_alphas.append(alpha_implicit)
            for h in hs_labels.numpy():
                all_labels.append(IDX2HATE.get(h, 'no'))

if all_alphas:
    alpha_matrix = np.vstack(all_alphas)
    df_gate = pd.DataFrame({
        'alpha_title': alpha_matrix[:, 0],
        'alpha_desc': alpha_matrix[:, 1],
        'hate_class': all_labels[:len(alpha_matrix)]
    })
    
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    sns.boxplot(x='hate_class', y='alpha_title', data=df_gate, palette='Set2')
    plt.title('Context Source Gate: Title Relevance (alpha_T)')
    plt.ylabel('alpha_Title Weight')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sns.boxplot(x='hate_class', y='alpha_desc', data=df_gate, palette='Set2')
    plt.title('Context Source Gate: Description Relevance (alpha_D)')
    plt.ylabel('alpha_Desc Weight')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("Mean Gating Weights by Class:")
    print(df_gate.groupby('hate_class')[['alpha_title', 'alpha_desc']].mean().round(3))

## 12. Trích Xuất Task C Bridge Representations $h_B$
Mô hình RACC cung cấp biểu diễn cầu nối $h_B = \sum_c p_c \cdot z_c \in \mathbb{R}^{B \times 768}$ kết hợp thông tin phân loại thù ghét có trọng số để truyền trực tiếp sang Task C (Stereotype Category Identification).

In [ ]:
from sklearn.manifold import TSNE

racc_model.eval()
all_hb = []
all_gold = []

with torch.no_grad():
    for batch in val_loader:
        ids, mask, roles, _, _, hs_labels, _ = batch
        ids = ids.to(trainer.device)
        mask = mask.to(trainer.device)
        roles = roles.to(trainer.device)
        _, h_b, _ = racc_model(ids, mask, roles, return_diagnostics=False)
        all_hb.append(h_b.cpu().numpy())
        for h in hs_labels.numpy():
            all_gold.append(IDX2HATE.get(h, 'no'))

hb_matrix = np.vstack(all_hb)
print(f"Extracted Task C Bridge Embeddings: shape {hb_matrix.shape}")

# Biểu diễn 2D t-SNE của Task C Bridge Embeddings
if len(hb_matrix) >= 10:
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(hb_matrix)-1))
    hb_2d = tsne.fit_transform(hb_matrix)
    
    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        x=hb_2d[:, 0], y=hb_2d[:, 1], 
        hue=all_gold[:len(hb_matrix)], 
        palette={'no': 'lightgray', 'yes_implicit': 'orange', 'yes_explicit': 'crimson'},
        alpha=0.8
    )
    plt.title('t-SNE Projection of Task C Bridge Representations (h_B)')
    plt.xlabel('t-SNE Dim 1')
    plt.ylabel('t-SNE Dim 2')
    plt.legend(title='Hate Speech Class')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 13. Xuất Kết Quả Dự Đoán & Checkpoint Chính Thức
Lưu checkpoint mô hình RACC tốt nhất và xuất file dự đoán TSV theo định dạng nộp bài chính thức của SemEval StereoQueerEval 2027.

In [ ]:
out_dir = "checkpoints_racc"
os.makedirs(out_dir, exist_ok=True)

# Lưu checkpoint mô hình
ckpt_path = os.path.join(out_dir, "task_b_racc_best.pt")
torch.save({
    'model_state_dict': racc_model.state_dict(),
    'config': config.to_dict(),
    'macro_f1': macro_f1,
    'f1_implicit': f1_dict.get('yes_implicit', 0.0)
}, ckpt_path)
print(f"Successfully saved best RACC checkpoint to {ckpt_path}")

# Xuất file CSV dự đoán
pred_csv = os.path.join(out_dir, "task_b_racc_val_predictions.csv")
df_analysis.to_csv(pred_csv, index=False)
print(f"Successfully exported validation predictions and diagnostics to {pred_csv}")